# LoRA / QLoRA supervised fine-tuning of a causal language model
Instruction-tunes (SFT) a decoder-only LLM — Llama, Mistral, Qwen, Gemma, Phi — with LoRA
adapters using Hugging Face `transformers` + `peft`. Set `use_qlora = True` to load the base model
in 4-bit (bitsandbytes) for large models on a single GPU.

Data: a JSONL/CSV file with either a `text` column, or `prompt` + `response` columns (chat
template applied when the tokenizer has one), or a Hugging Face dataset id in `hf_dataset`.
Output: the LoRA adapter + tokenizer (optionally a merged model), train/eval metrics. Training
metrics are logged to MLflow through the Trainer's MLflow callback.

Requires: torch, transformers, peft, datasets, accelerate (+ bitsandbytes for QLoRA) and a GPU.

## Parameters

In [ ]:
base_model = "meta-llama/Llama-3.2-1B-Instruct"   # Hugging Face model id or local path
dataset_path = ""            # JSONL/CSV file (staged by the platform) - or use hf_dataset
hf_dataset = ""              # e.g. "tatsu-lab/alpaca" when no dataset_path is given
hf_split = "train"
text_field = "text"          # single-text column (used if present)
prompt_field = "prompt"      # else prompt + response columns
response_field = "response"
use_qlora = False            # 4-bit base model (needs bitsandbytes)
lora_r = 16
lora_alpha = 32
lora_dropout = 0.05
target_modules = "all-linear"  # or a comma list, e.g. "q_proj,v_proj"
num_train_epochs = 1.0
max_steps = -1               # >0 overrides epochs
learning_rate = 2e-4
per_device_train_batch_size = 4
gradient_accumulation_steps = 4
max_seq_length = 1024
eval_fraction = 0.05
logging_steps = 10
bf16 = True
seed = 42
merge_adapter = False        # also save a merged full model (needs disk + memory)
output_dir = "outputs"

## Setup

In [ ]:
import json, os
output_dir = os.environ.get("STUDIO_OUTPUT_DIR", output_dir) if output_dir == "outputs" else output_dir
os.makedirs(output_dir, exist_ok=True)
import torch
from datasets import load_dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer, DataCollatorForLanguageModeling,
                          Trainer, TrainingArguments, set_seed)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
set_seed(seed)
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
report_to = ["mlflow"] if os.environ.get("MLFLOW_RUN_ID") else []

## Load data

In [ ]:
if dataset_path:
    fmt = "csv" if dataset_path.endswith(".csv") else "json"
    ds = load_dataset(fmt, data_files=dataset_path, split="train")
elif hf_dataset:
    ds = load_dataset(hf_dataset, split=hf_split)
else:
    raise ValueError("Set dataset_path or hf_dataset")
print(ds)

## Tokenizer and formatting

In [ ]:
tok = AutoTokenizer.from_pretrained(base_model, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def to_text(ex):
    if text_field in ex and ex[text_field]:
        return ex[text_field]
    p, r = ex.get(prompt_field, ""), ex.get(response_field, "")
    if getattr(tok, "chat_template", None):
        return tok.apply_chat_template([{"role": "user", "content": p},
                                        {"role": "assistant", "content": r}], tokenize=False)
    return f"### Instruction:\n{p}\n\n### Response:\n{r}{tok.eos_token}"

def tokenize(ex):
    return tok(to_text(ex), truncation=True, max_length=max_seq_length)

ds = ds.map(tokenize, remove_columns=ds.column_names)
split = ds.train_test_split(test_size=eval_fraction, seed=seed) if eval_fraction > 0 else {"train": ds, "test": None}
train_ds, eval_ds = split["train"], split["test"]
print("train", len(train_ds), "eval", len(eval_ds) if eval_ds is not None else 0)

## Model with LoRA adapters

In [ ]:
kw = {"torch_dtype": torch.bfloat16 if bf16 else torch.float16}
if use_qlora:
    from transformers import BitsAndBytesConfig
    kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                   bnb_4bit_compute_dtype=kw["torch_dtype"],
                                                   bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(base_model, device_map="auto", **kw)
if use_qlora:
    model = prepare_model_for_kbit_training(model)
tm = target_modules if target_modules == "all-linear" else [m.strip() for m in target_modules.split(",")]
model = get_peft_model(model, LoraConfig(r=lora_r, lora_alpha=lora_alpha, lora_dropout=lora_dropout,
                                         target_modules=tm, task_type="CAUSAL_LM"))
model.print_trainable_parameters()

## Train

In [ ]:
args = TrainingArguments(
    output_dir=os.path.join(output_dir, "checkpoints"), num_train_epochs=num_train_epochs,
    max_steps=max_steps, learning_rate=learning_rate,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps, logging_steps=logging_steps,
    eval_strategy="epoch" if eval_ds is not None else "no", save_strategy="no",
    bf16=bf16, fp16=not bf16 and torch.cuda.is_available(), report_to=report_to, seed=seed,
    lr_scheduler_type="cosine", warmup_ratio=0.03, gradient_checkpointing=True)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
                  data_collator=DataCollatorForLanguageModeling(tok, mlm=False))
train_result = trainer.train()
trainer.save_metrics("train", train_result.metrics)
print(train_result.metrics)

## Evaluate and save

In [ ]:
if eval_ds is not None:
    ev = trainer.evaluate()
    trainer.save_metrics("eval", ev)
    print(ev)
adapter_dir = os.path.join(output_dir, "adapter")
model.save_pretrained(adapter_dir)
tok.save_pretrained(adapter_dir)
if merge_adapter:
    merged = model.merge_and_unload()
    merged.save_pretrained(os.path.join(output_dir, "merged"), safe_serialization=True)
    tok.save_pretrained(os.path.join(output_dir, "merged"))
import shutil; shutil.rmtree(os.path.join(output_dir, "checkpoints"), ignore_errors=True)
print("saved to", output_dir)